In [ ]:
"""
    주  제: Python을 활용한 금융데이터 분석 교육 자료
    작성자: NICE홀딩스 디지털전략본부 박재휘
    작성일: 2022.04.03
"""

# shift + endeer : interactive로 선택된 부분만 실행
# cmd로 실행 후 관련 라이브러리 설치 (pip install)

# 금융 raw 데이터 형태
# 1. csv (,로 나뉨)
# 2. txt (구분자 : tab)
# 3. txt (구분자 : | )
# 4. txt (고정길이, 데이터 만든 사람만 규칙을 앎)

# -> 파이썬으로 불러올 때 read_csv 매서드 가장 많이 사용

import pandas as pd
import numpy as np
import datetime as dt


###############################################################################
# Utils
###############################################################################
def read_sas(file_name: str, format: str = 'sas7bdat', verbose: bool = False):
    """ sas7bdat 파일을 읽고, binary까지 변환해주는 함수 """
    infiled = pd.read_sas(file_name, format=format)
    for col in infiled.columns:
        if verbose:
            print(infiled[col].dtype, end=' ')
        if pd.api.types.is_string_dtype(infiled[col]):
            if verbose:
                print(pd.api.types.is_string_dtype(infiled[col]))
            infiled[col] = infiled[col].str.decode('CP949')
        elif verbose:
            print(False)
    return infiled



###############################################################################
# 1. pandas 라이브러리 기본 이해
###############################################################################
# 0.1 Data Infile
# 일반 데이터 파일을 python에서 분석하기 위해서는 데이터를 읽어오는 방법 필요
pop1 = pd.read_csv('../data/credit/Data/pop.csv')   # ..로 한 단계 올라가서 경로 설정
score = pd.read_csv('../data/credit/Data/SCORE.txt', sep='\t')  # read_csv + sep으로 txt도 처리 가능
cps = pd.read_csv('../data/credit/Data/CPS.txt', sep='|')
cb_delq_raw = pd.read_fwf('../data/credit/Data/CB_DELQ_RAW.txt',
                          header=None,  # 헤더 정보는 없다
                          widths= [5,8,8,9])    # rule 기반
# cb_delq_raw 파일은 header 정보가 없으므로, 따로 정의
cb_delq_raw = cb_delq_raw.set_axis(['NO', 'FST_OCR_DATE', 'REL_DATE', 'DQ_AMT'],    # set_axis로 헤더 정보 추가
                                   axis=1)
# from SAS
vintage_key = read_sas('../data/credit/Data/vintage_key.sas7bdat', verbose=True)
rollrate_key = read_sas('../data/credit/Data/rollrate_key.sas7bdat', verbose=True)
dlq = read_sas('../data/credit/Data/dlq.sas7bdat', verbose=True)


# 1.1 pandas DataFrame  
## pandas DataFrame은 Data Table을 쉽게 처리하는 도구
# pandas는 Series / DataFrame 클래스를 이용하여 데이터를 처리하는 라이브러리
print(pop1)
# DataFrame, Series
# DataFrame : 전체  (= 여러 series의 조합임)
# Series : 하나의 열
print(type(pop1['NO'])) # 불러올 열을 대괄호[]로 넣기 => series
print(type(pop1))       # => dataframe

# 1.2 pandas DataFrame format
# 큰 기준의 데이터 타입: 문자, 숫자, 날짜
# object: 문자
# int/float: 숫자
# datetime: 날짜
print(pop1.dtypes) # 원래 의미하는 데이터 타입대로 불러왔는지 확인할 때 사용
pop1['EW0020901'] = pop1['EW0020901'].astype(str)

# 1.3 summary
# 데이터 간략 요약
pop1.describe()                     # 숫자형            
pop1.describe(include=np.number)    # 숫자형

pop1.describe(include='O')          # 문자형
pop1.describe(include='object')     # 문자형

pop1.describe(include='all')        # 숫자형 + 문자형

# 1.4 데이터 확인
pop1.head()     # 앞부분 확인, default=5
pop1.tail()     # 뒷부분 확인
pop1.head(10)

# 1.5 format 변경
# 1.5.1 숫자를 문자형으로 변경  (바꾸기만하고 할당을 안함!)
pop1['EW0020901'].map(str)
pop1['EW0020901'].apply(str)
pop1['EW0020901'].astype(str)   # 젤 많이 씀
pop1.dtypes
# 실제 변경(할당)
pop1['EW0020901'] = pop1['EW0020901'].astype('object')
pop1.dtypes

# 1.5.2 문자를 숫자로 변경
pd.to_numeric(pop1['EW0020901'])
pop1['EW0020901'].apply(pd.to_numeric)
pop1['EW0020901'].astype('int')     # 정수
pop1['EW0020901'].astype('float')   
pop1['EW0020901'] = pd.to_numeric(pop1['EW0020901'])    # 할당!

# 1.5.3 날짜형으로 변경
pd.to_datetime(pop1.stday)  # 이대로 하면 X
pd.to_datetime(pop1.stday, format="%Y%m%d") # format을 넣기!



###############################################################################
# 2. pandas 라이브러리 기본 함수
###############################################################################
# 2.1 문자함수
# 2.1.0 sample data
sample = pd.DataFrame({'mail': ['iamhwii@gmail.com',
                                'jhp@nice.co.kr'],
                       'name': ['JaeHwi ', '  Park  '],     # 공백있음.
                       'job_code': ['1010201', '3010499']})

# 2.1.1 substr, left, right
sample['mail'].str[3:7] # => hwii, @nic
sample['mail'].str[:4]  # => 왼쪽에 있는 4개
sample['mail'].str[5:]  # => 오른쪽에 있는 5개
sample['mail'].str[-3:]  # => 오른쪽에 있는 3개

# 2.1.2 index, find
sample['mail'].str.index('@')   
sample['mail'].str.find('@')
sample['mail'].str.index('Z')   # 없으면 error
sample['mail'].str.find('Z')    # 없으면 -1
sample['mail'].str.findall('i')     # 모든 i를 리스트로 찾아줌
sample['mail'].str.findall('..')    # 2개씩 찾기 (. = 임의의 글자)
sample['mail'].str.findall('Z') # 없으면 빈 list
# 2.1.3 count
sample['mail'].str.count('i')   
sample['mail'].str.count('.')   # 총 개수 (. = 임의의 글자)
sample['mail'].str.count('\.')  # 실제 .을 찾기 위해 \ 넣기
sample['mail'].str.count('Z')   # 없으면 0
# 2.1.4 len
sample['name']
sample['name'].str.len()        # 공백을 고려한 길이

# 2.1.5 strip
sample['name']
sample['name'].str.strip()      # 공백을 없애는 작업    
sample['name'].str.lstrip("JP") # 왼쪽 공백 + JP 제거 (공백 주의!)
sample['name'].str.lstrip("JaP")    
sample['name'].str.lstrip("JaP ")
sample['name'].str.rstrip("JP") # 오른쪽

# 2.1.6 concat
sample['mail'] + sample['name'] # 연결
# 2.1.7 replace
sample['mail'].str.replace('@', '#')    # @ -> # 로 대체 

# 2.1.8 contains
sample['job_code'].str.contains('30')   # 30글자가 있는지 -> T/F
# 2.1.9 startswith
sample['job_code'].str.startswith('30') # 30으로 시작하는지 -> T/F
# 2.1.10 endswith
sample['job_code'].str.endswith('99')   # 30으로 끝나는지 -> T/F

# 2.1.11 lower, upper
sample['mail'].str.lower()  # all 소문자로
sample['mail'].str.upper()  # all 대문자로

# 2.1.12 split
sample['mail'].str.split("@")   # @를 기준으로 list로 나누기
# 2.1.13 isnumeric  (문자열 데이터지만 숫자로 찾을 수 있느냐 -> T/F )
sample['mail'].str.isnumeric()  
sample['job_code'].str.isnumeric()
sample['name'].str.isnumeric()
# 2.1.13 isalpha    (문자열 데이터지만 알파벳으로 찾을 수 있느냐 -> T/F )
sample['mail'].str.isalpha()
sample['job_code'].str.isalpha()
sample['name'].str.isalpha()    # 공백있어서 F
pd.Series(['abc', 'BBad', 'c1d2']).str.isalpha()    # 오직 알파벳만 있어야 T


# 2.2 날짜함수
# 2.2.0 sample data
sample2 = cb_delq_raw[['FST_OCR_DATE', 'REL_DATE']].drop_duplicates()
sample2.loc[sample2['REL_DATE']==99991231, 'REL_DATE'] = 20301231
sample2 = sample2.apply(pd.to_datetime, format='%Y%m%d')
sample2.dtypes

# 2.2.1 year, month, day
sample2['FST_OCR_DATE'].dt.year     # 연도만 뽑기
sample2['FST_OCR_DATE'].dt.month    # 월만 뽑기
sample2['FST_OCR_DATE'].dt.day      # 일자만 뽑기

# 2.2.2 weekday: Monday=0, Sunday=6
sample2['FST_OCR_DATE'].dt.weekday  # 숫자로 요일 표현(0=월요일,..., 6=일요일) 
sample2['FST_OCR_DATE'].dt.day_name()   # 문자로 요일 표현 (직관적이지만 문자가 더 다룰 때 편함)

# 2.2.3 is_month_start, is_month_end            ( 월의 시작인지, 끝인지 -> T/F)
sample2['FST_OCR_DATE'].dt.is_month_start
sample2['FST_OCR_DATE'][sample2['FST_OCR_DATE'].dt.is_month_start]  # 월의 시작인 것만(T만) 출력
sample2['FST_OCR_DATE'].dt.is_month_end 
sample2['FST_OCR_DATE'][sample2['FST_OCR_DATE'].dt.is_month_end]    # 월의 끝인 것만(T만) 출력

# 2.2.4 quarter
sample2['FST_OCR_DATE'].dt.quarter  # 분기 표현
# 2.2.5 is_quarter_start, is_quarter_end        ( 분기의 시작인지, 끝인지 -> T/F)
sample2['FST_OCR_DATE'].dt.is_quarter_start 
sample2['FST_OCR_DATE'][sample2['FST_OCR_DATE'].dt.is_quarter_start]    # 분기의 시작인 것만(T만) 출력 
sample2['FST_OCR_DATE'].dt.is_quarter_end
sample2['FST_OCR_DATE'][sample2['FST_OCR_DATE'].dt.is_quarter_end]      # 분기의 끝인 것만(T만) 출력

# 2.2.6 days_in_month
sample2['FST_OCR_DATE'].dt.days_in_month    # 한 달에 며칠 있는지

# 2.2.7 datetime: today, now  =>  지금 시간
dt.datetime.today() 
dt.datetime.now()   

# 2.2.8 hour, minute, second    현재 시간을 시,분,초 Series로 뽑기
pd.Series(dt.datetime.today()).dt.hour
pd.Series(dt.datetime.today()).dt.minute
pd.Series(dt.datetime.today()).dt.second

# 2.2.9 날짜 차이 계산  (days, seconds만 있음)
diff = dt.datetime(2022, 1, 1,12,0,0) - dt.datetime(2021, 12, 31,1,0,0)
diff.days
diff.seconds
(dt.datetime(2022, 1, 1) - dt.datetime(2021, 12, 1)).days   
    # 양편 넣기 -> 32일 / 한편 넣기 -> 31일

# 2.2.10 날짜 이동
dt.datetime(2022, 1, 1) + dt.timedelta(days=-1) # 하루 빼기 (# dt.timedelta는 변동)
dt.datetime(2022, 1, 1) + dt.timedelta(weeks=1) # 일주일 더하기 
dt.datetime(2022, 1, 1) + pd.DateOffset(months=1)   # 한 달 더하기 (# pd.DateOffset는 타임스템프 방식으로 바뀜)
one_year_later = dt.datetime(2022, 1, 1) + pd.DateOffset(years=1)
one_year_later.to_pydatetime()  # to_pydatetime으로 datetime으로 출력 형식 바꿈
# FST_OCR_DATE를 날짜 이동
sample2['FST_OCR_DATE'] + dt.timedelta(days=-1) # 하루 빼기
sample2['FST_OCR_DATE'] + dt.timedelta(weeks=1) # 일주일 더하기 
sample2['FST_OCR_DATE'] + pd.DateOffset(months=1)   # 한 달 더하기
sample2['FST_OCR_DATE'] + pd.DateOffset(years=1)    # 1년 더하기 

# 2.2.11 날짜 포맷 변환
sample2['FST_OCR_DATE'].dt.strftime('%Y-%m-%d') # 시간 -> 문자형으로 (strftime)
pd.to_datetime(cb_delq_raw['FST_OCR_DATE'])     # 숫자 -> 시간
dt.datetime.strptime('2022-01-01', '%Y-%m-%d')  # 문자형 -> 시간 (strptime)


# axis=0 : row 행
# axis=1 : col 열

# 2.3 숫자함수: sum, min, max, mean, count, median
# 2.3.0 sample data
sample3 = cps.iloc[:6][['CF0600217','CF0600218','CF0600224']]
# 2.3.1 sum
sample3.sum()       # default : axis=0 행 더하기 
sample3.sum(axis=1) # 열 더하기
# 2.3.2 min
sample3.min()
sample3.min(axis=1)
# 2.3.3 max
sample3.max()
sample3.max(axis=1)
# 2.3.4 mean
sample3.mean()
sample3.mean(axis=1)
# 2.3.5 count
sample3.count()
sample3.count(axis=1)


# 2.4 기타 함수
# 2.4.0 sample data
sample4 = pd.DataFrame(dict(age=[5, 6, np.NaN],
                            born=[pd.NaT, pd.Timestamp('1939-05-27'),
                                  pd.Timestamp('1940-04-25')],
                            name=['Alfred', 'Batman', ''],      # 공백 있음. 
                            toy=[None, 'Batmobile', 'Joker']))
sample5 = pop1[['NO','stday']]
# 2.4.1 isna, isnull : 값이 할당되지 않은 것 (이 경우 공백은 정보 있는 것임.) 
sample4.isna()  
sample4.isna() == sample4.isnull()  # 둘이 같은 함수임
# 2.4.2 notna, notnull : 값이 할당된 것     (<=> isna, isnull) 정반대임
sample4.notna()
sample4.notna() == sample4.notnull()
# 2.4.3 dropna : na인걸 drop해라. 
sample4.dropna()    # 즉, 모든 열에 값이 할당된 것만 남겨라. 
sample4.dropna(axis=1)  # 즉, 모든 행에 값이 할당된 것만 남겨라. 
sample4.dropna(subset=['age'])  # age가 null인 행만 지워라
# 2.4.4 fillna
sample4.fillna(-999)    # 빈 값들에 특수값 채우기 

# 2.4.5 duplicated  
sample5.duplicated()    # 중복 판단
sample5.duplicated().sum()  # 중복 개수
sample5.duplicated(subset=['NO','stday']).sum() 

# 동일 --
sample5.duplicated(subset='NO').sum()   
sample5['NO'].duplicated().sum()
# -------

# 동일 --
sample5.duplicated(subset='stday').sum()    # 날짜만 중복인 것 개수 
sample5['stday'].duplicated().sum()
# -------

# 2.4.6 drop_duplicates : 중복 drop
sample5.drop_duplicates()
sample5.drop_duplicates(subset='NO')                    # 첫번째꺼는 안잡힘 -> 18497
sample5[sample5.duplicated(subset='NO', keep=False)]    # keep=False : 모두 잡힘 -> 2715
sample5.drop_duplicates(subset='NO', keep=False)        # 모든 중복 다 날림 -> 17285 (<=> sample5[sample5.duplicated(subset='NO', keep=False)] 정반대임)
sample5['NO']

# 2.4.7 value_counts
score.RK0600_700_GRD.value_counts()     # RK0600_700_GRD 각 케이스에 대한 건 수 (건 수로 내림차순 정렬)
# ( = score['RK0600_700_GRD'].value_counts())
score.RK0600_700_GRD.value_counts(normalize=True)   # 비율로 
score.RK0600_700_GRD.value_counts(sort=False)       # (건 수로 정렬 X)
score.RK0600_700_GRD.value_counts(sort=False).sort_index()  # (건 수로 정렬 X, index로 정렬)
score.RK0600_700_GRD.value_counts(sort=True, ascending=True)    # 올림차순 정렬
score.RK0600_700_SCR.value_counts(bins=10)  # 구간별로 
score.RK0600_700_GRD.value_counts(dropna=False) # null값도 같이 세기

# 2.4.8 set_axis
cb_delq_raw = pd.read_fwf('../data/credit/Data/CB_DELQ_RAW.txt',
                          header=None,
                          widths= [5,8,8,9])
cb_delq_raw = cb_delq_raw.set_axis(['NO', 'FST_OCR_DATE', 'REL_DATE', 'DQ_AMT'], 
                                   axis=1)  # axis=1로 열 정보 추가 
cb_delq_raw.set_axis(cb_delq_raw['NO'], axis=0) # NO를 축으로 쓰기 + 겹침 문제
cb_delq_raw.set_axis(cb_delq_raw['NO'], axis=0).drop('NO', axis=1)  # 겹침 문제 해결
# 2.4.9 set_index
cb_delq_raw.set_index('NO') # 바로 NO를 축으로 설정
cb_delq_raw.set_index(cb_delq_raw['NO'])



###############################################################################
# 3. pandas 라이브러리 활용1
###############################################################################
# 3.1 열 선택
pop1['stday'] # type: Series
pop1.stday  # 되는 경우, 안되는 경우(ex 스페이스나 , ) 있어서 위에 방법 추천 
pop1[['stday']] # type: DataFrame
pop1[['NO', 'stday', 'EW0020901']]
pop1.loc[['NO', 'stday', 'EW0020901']]  # 행에 대한 정보가 없어서 error
pop1.loc[:, ['NO', 'stday', 'EW0020901']]   # 행 + 열에 대한 정보 있어야함. (: 전체 열)
pop1.iloc[:, 1] # 이름이 아닌 위치로 꺼내기
pop1.iloc[:, 1:4]
pop1.iloc[:, [0,1,2]]
pop1.drop(['DQ0052001', 'P27000100'], axis=1)   # 원하지 않는 열을 빼고 출력

# 3.2 행 선택
pop1[pop1['NO'] == 1]       # NO가 1인 행 선택
pop1['NO'] == 1 # -> T/F
pop1[[False, True, True] + [False] * 19997]     # 2,3만 출력됨.
pop1.loc[[False, True, True] + [False] * 19997]
pop1.loc[[False, True, True] + [False] * 19997, 'NO']   # type: Series
pop1.loc[[False, True, True] + [False] * 19997, ['NO', 'stday', 'EW0020901']]
pop1.loc[[False, True, True] + [False] * 19997, ['NO']] # type: DataFrame
pop1.iloc[1:3, 1]
pop1.iloc[1:3, 1:3]
pop1.drop([1,2])

# 3.3 조건 적용
# np.where
pop1['COND'] = np.where(pop1['NO'] > 3, True, False)    # NO가 3보다 크면 T, 아니면 F -> COND에 저장
# lambda 함수
pop1['COND'] == pop1['NO'].apply(lambda x: x > 3)   # 위 방법과 동일한 결과 나옴
# 일반 함수             위 방법과 동일한 결과 나옴
def temp_function(x):
    if x > 3:
        return True
    else:
        return False
pop1['COND'] == pop1['NO'].apply(temp_function)

# 3.4 정렬
pop1
pop1.sort_values(by='stday')    # 날짜로 정렬
pop1.sort_values(by=['stday', 'EW0020901'])     # 날짜로 정렬 + 날짜가 같다면 EW0020901로 정렬
pop1.sort_values(by='stday', ascending=False)   # ascending=False : 내림차순

# 3.5 병합
# 3.5.0 sample data
df1 = pd.DataFrame({
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    }, index=[1, 2, 3, 4])
df2 = pd.DataFrame({
        "A": ["A4", "A5", "A6", "A7"],
        "B": ["B4", "B5", "B6", "B7"],
        "C": ["C4", "C5", "C6", "C7"],
        "D": ["D4", "D5", "D6", "D7"],
    }, index=[2, 3, 6, 7])
df3 = pd.DataFrame({
        "A": ["A8", "A9", "A10", "A11"],
        "B": ["B8", "B9", "B10", "B11"],
        "C": ["C8", "C9", "C10", "C11"],
        "D": ["D8", "D9", "D10", "D11"],
        "E": ["E8", "E9", "E10", "E11"],
    }, index=[3, 4, 6, 7])
left = pd.DataFrame({
        "key": ["K0", "K1", "K2", "K3"],
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
    })
right = pd.DataFrame({
        "key": ["K0", "K1", "K3", "K4"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    })
# 3.5.1 concat
# 세로
frames = [df1, df2, df3]
pd.concat(frames)       # 인덱스도 별다른 작업 없이 그냥 세로로 합침 + 모든 열 합집합
pd.concat(frames, ignore_index=True)    # 인덱스 재세팅
pd.concat(frames, ignore_index=True, join='inner')  # 교집합 열만 넣어짐
# 가로
pd.concat(frames, axis=1)   # 가로로 그냥 합침 + 열집합은 합집합으로 
pd.concat(frames, axis=1, join='inner') # 교집합 행만 넣어짐
pd.concat(frames, axis=1, ignore_index=True)    # 행 정보 재세팅됨 

# 3.5.2 merge   (가로로 병합하는 방법)
pd.merge(left, right, on="key") # 교집합인 키를 기준으로 left, right 순서대로 병합
pd.merge(left, right, on="key", how='outer')    # 합집합으로 
pd.merge(left, right, on="key", how='left')     # left 기준 합집합으로 (가장 많이 사용)
pd.merge(left, right, on="key", how='right')    # right 기준 합집합으로 
pd.merge(left, right, left_on="key", right_on="key")    # 양 쪽 기준 달라질 때 사용하면 유용
# 중복체크  
pd.merge(pd.DataFrame({"A": [1, 2], "B": [1, 2]}), 
         pd.DataFrame({"A": [4, 5, 6], "B": [2, 2, 2]}), 
         on="B", how="outer", validate="one_to_one")    # 일부로 1:1이 아니면 에러를 발생시켜 중복확인
pd.merge(pd.DataFrame({"A": [1, 2], "B": [1, 2]}), 
         pd.DataFrame({"A": [4, 5, 6], "B": [2, 2, 2]}), 
         on="B", how="outer", indicator=True)           # 에러는 아니고 추가적인 정보 컬럼을 붙임

# 3.5.3 join    (가로로 병합하는 방법)
A = left.set_index('key')   # 키를 인덱스로 
B = right.set_index('key')  # 키를 인덱스로 
A.join(B)   # A를 기준으로 B를 붙이기 
A.join(B, how='outer')  # 합집합
A.join(B, how='inner')  # 교집합

# 3.6 함수 적용
# 3.6.0 sample data
sample_df = pd.DataFrame([[4, 9]] * 3, columns=['A', 'B'])
# 3.6.1 apply 기본 
sample_df.apply(np.sqrt)    # 각각에 제곱근
sample_df.apply(np.sum)     # 합 (axis=0)
sample_df.apply(np.sum, axis=1)
# 3.6.2 apply customizing1
sample_df.apply(lambda ss: 2*ss + 1)
sample_df.apply(lambda x: x.sum() + 1)  # 합 (axis=0)
sample_df.apply(lambda x: x.sum() + 1, axis=1)
# 3.6.3 apply customizing2
def myfunction(x):
    if x.sum() < 15:
        return x/2
    else:
        return x*3
sample_df.apply(myfunction)



###############################################################################
# 4. pandas 라이브러리 활용2 (groupby)
###############################################################################
# 4.1 DataFrameGroupBy object 생성
grouped = score.groupby('RK0600_700_GRD')
# DataFrameGroupBy object는 Iterable
for name, group in grouped:     # -> 10개의 구간으로 그룹화함
    print(name)
    display(group) # notebook의 display() 함수

# 4.2 집계
# 4.2.1 집계 방법1 -> {열:연산} dict를 입력
score.groupby('RK0600_700_GRD').agg({'NO': 'count'})    # -> 데이터프레임
# 4.2.2 집계 방법2 -> 컬럼 선택후 연산자만 전달
score.groupby('RK0600_700_GRD')['NO'].agg('count')      # -> 시리즈
# 4.2.3 집계 방법2 -> agg 메서드 없이 바로 연산 함수 활용 가능
score.groupby('RK0600_700_GRD')['NO'].count()           # -> 시리즈

# 4.3 다수 grouping
# 4.3.1 2개 columns 그룹핑
score.groupby(['RK0600_700_GRD', 'ML0100_001_GRD'])['NO'].count()   # 두 개의 컬럼 기준
# 4.3.2 2개 columns 그루핑, 2개 columns 집계
pop1.groupby(['NO', 'stday'])[['B12000200', 'P27000100']].sum()
# 4.3.3 2개 columns 그루핑, 2개 columns, 두 가지 집계
pop1.groupby(['NO', 'stday'])[['B12000200', 'P27000100']].agg(['sum', 'count'])
# 4.3.4 2개 columns 그룹, 2개 columns, 다양한 집계
agg_dict = {'RK0600_700_SCR':['sum', 'mean', 'size'],
            'ML0100_001_GRD':['sum', 'var']}
score.groupby('stday').agg(agg_dict)

# 4.4 사용자 정의 집계
# 4.4.1 사용자 정의 집계 함수
def my_func(s):
    return s.max()
# 4.4.2 집계 Column명 수정 가능
my_func.__name__ = '그룹의 최대값 함수'
# 4.4.3 사용자 정의 함수를 agg에 활용
score.groupby('stday')[['RK0600_700_SCR', 'ML0100_001_SCR']].agg(my_func)
# 4.4.4 사용자 정의 함수를 apply에 활용
score.groupby('stday')[['RK0600_700_SCR', 'ML0100_001_SCR']].apply(my_func)
# 테스트
score.loc[score['stday'] == 20190407, ['RK0600_700_SCR', 'ML0100_001_SCR']]

# 4.5 범주형 변환
bins = [-np.inf, 400, 500, 600, 700, 800, np.inf]   # -> 6개의 구간
labels=['400이하','500이하','600이하','700이하','800이하','800초과']
cuts = pd.cut(score['RK0600_000_SCR'], bins=bins, labels=labels, right=True)    # 오른쪽을 포함 (]
# 해당 범주를 그룹 기준으로 활용
score.groupby(cuts)['NO'].count()

# 4.6 pivot table
score.pivot_table(index='RK0600_700_GRD',  # axis 0
                 columns='ML0100_001_GRD', # axis 1
                 values='NO',     # 집계항목
                 aggfunc='count', # 집계방법
                 fill_value=0)    # null인경우

# 4.7 transform
# 4.7.1 sample data
sample6 = pd.DataFrame({
        "key": ["K0", "K1", "K0", "K1", "K0", "K1"],
        "A": ["1", "3", "4", "6", "7", "9"],
        "B": ["2", "5", "8", "9", "10", "11"]
})
# 4.7.2 without transform
tmp = sample6.groupby("key").max()          # 축약된 상태로 집계
sample6.groupby("key")['B'].max()
sample6.join(tmp, on="key",rsuffix="_r")    # 원본 상태에 추가 
# 4.7.3 with transform
tmp2 = sample6.groupby("key").transform('max')  # 축약되지 않은 상태로 집계
pd.concat([sample6, tmp2], axis=1)



###############################################################################
# 4. 금융 데이터 분석 실습1
###############################################################################
""" 실습 4.1
    
    1) 실습 데이터: pop1
    2) 실습 과제:
        - stday 항목으로 기준년도(YYYY) 생성
        - stday 항목으로 기준월(MM) 생성
        - stday 항목으로 기준년월(YYYYMM) 생성
        - 직업코드 항목으로 직업군 항목 생성: pop1['EW0020901']
        - 직업코드 첫번째자리 의미: 1(급여소득자), 2(개인사업자), 3(기타소득자)
"""
# 4.1
# stday 항목으로 기준년도(YYYY) 생성
pop1['stday'].dtype
pop1['stday'].astype(str).str[:4]   # 문자로 바꿔서 뽑기
pd.to_datetime(pop1['stday'], format="%Y%m%d").dt.year  # 날짜형으로 바꿔서 뽑기
# stday 항목으로 기준월(MM) 생성
pop1['stday'].astype(str).str[4:6]
pd.to_datetime(pop1['stday'], format="%Y%m%d").dt.month
# stday 항목으로 기준년월(YYYYMM) 생성
pop1['stday'].astype(str).str[:6]
pd.to_datetime(pop1['stday'], format="%Y%m%d").dt.strftime('%Y%m')  # 연과 월을 한 번에 뽑기 

pop1['JOB_GRP'] = pop1['EW0020901'].astype(str).str[0]
pop1['JOB_GRP'] = pop1['JOB_GRP'].map({
    '1': '급여소득자',
    '2': '개인사업자',
    '3': '기타소득자'
})

""" 실습 4.2

    1) 실습 데이터: pop1
    2) 실습 과제:
        - 기준일자(stday) 이후 365일째 날짜
        - 기준일자(stday) 이후 12개월째에 해당되는 달의 말일 날짜
        - 최초연체발생일(FST_OCR_DATE)과 연체해제일(REL_Date) 기준으로 연체일수 계산
"""

# 4.2
# 기준일자(STDAY) 이후 365일째 날짜
pop1['STDAY_DT'] = pd.to_datetime(pop1['stday'], format="%Y%m%d")   # 날짜로 바꿔 STDAY_DT에 저장
pop1['STDAY_AF_365D'] = pop1['STDAY_DT'] + pd.DateOffset(days=365)  # 365일 뒤 
pop1['STDAY_AF_1Y'] = pop1['STDAY_DT'] + pd.DateOffset(years=1)     # 365일이 아닐 수도. (2월 같은 경우)
pop1.loc[pop1['STDAY_AF_1Y'] != pop1['STDAY_AF_365D'], 
         ['STDAY_DT', 'STDAY_AF_365D', 'STDAY_AF_1Y']]

# 기준일자(stday) 이후 12개월째에 해당되는 달의 말일 날짜
pop1['STDAY_AF_12M_END'] = pop1['STDAY_DT'] + \
                           pd.DateOffset(months=12) + \
                           pd.offsets.MonthEnd(0)   # 그 달의 말일로 맞춤
pop1['STDAY_AF_12M_END2'] = pop1['STDAY_DT'] + pd.offsets.MonthEnd(13)  # 이렇게 쓰면 안됨. 기준일이 월말인지/월 중인지에 따라 결과가 달라질 수 있음.
pop1.loc[pop1['STDAY_AF_12M_END'] != pop1['STDAY_AF_12M_END2'],
         ['stday', 'STDAY_AF_12M_END', 'STDAY_AF_12M_END2']]

# 최초연체발생일(FST_OCR_DATE)과 연체해제일(REL_Date) 기준으로 연체일수 계산
cb_delq_raw['FST_OCR_DATE'] = pd.to_datetime(cb_delq_raw['FST_OCR_DATE'], 
                                             format="%Y%m%d")
cb_delq_raw['REL_DATE_N'] = cb_delq_raw['REL_DATE'].astype(str) # REL_DATE를 문자열로 변환
cb_delq_raw.loc[cb_delq_raw['REL_DATE_N']=='99991231', 'REL_DATE_N'] \
    = (dt.datetime.today() + dt.timedelta(days=1)).strftime('%Y%m%d')   # 계산 가능한 날짜로 대체 -> 오늘 날짜 + 1일로 치환
cb_delq_raw['REL_DATE_N'] = pd.to_datetime(cb_delq_raw['REL_DATE_N'], format="%Y%m%d")  # 날짜형으로 바꿈

cb_delq_raw.loc[cb_delq_raw['REL_DATE_N'] == '2022-04-24']
cb_delq_raw['DQ_DAYS'] = (cb_delq_raw['REL_DATE_N'] - cb_delq_raw['FST_OCR_DATE']).dt.days
print(cb_delq_raw['DQ_DAYS'].describe())




###############################################################################
# 5. 금융 데이터 분석 실습2
###############################################################################
""" 실습 5.1

    1) 실습 데이터: 
        - pop1
        - score
    2) 실습 과제:
        - 아래 요건에 따라 우선순위별 제외 사유코드 생성
            우선순위1 : (신용관리대상/공공정보)미해제등록 총건수(B12000200)  1건 이상
            우선순위2 : (채무불이행-신용정보사)미해제등록 총건수(B22000200)  1건 이상
            우선순위3 : 현재 개인회생신청정보 미해제 여부(BE0000801)  여
            우선순위4 : 대부업계 30일이상 연체 미해제 총건수(DQ0052001)  1건 이상
            우선순위5 : 미해제 연체 총 건수(P27000100)  1건 이상
            우선순위6 : 1~5 모두 해당하지 않으면 정상
        - 정상건만 추출하여 개발모집단 데이터셋(pop_dev) 생성
        - TIP스코어등급(ML0100_001_GRD)과 중금리ML스코어등급 (ML0100_002_GRD)을 
          아래와 같이 결합하여 결합등급을 생성
                     ML0100_002_GRD 1 ~ 10등급

                M    1 1 2 3 4 5  6  7  8  9
                L    1 2 2 3 4 5  6  7  8  9
                0    2 2 3 4 4 5  6  7  8  9
                1    3 3 4 4 5 5  6  7  8  9
                0    4 4 4 5 5 6  6  7  8  9
                0    5 5 5 5 6 6  7  7  8  9
                |    6 6 6 6 6 7  7  8  8 10
                0    7 7 7 7 7 7  8  8 10 10
                0    8 8 8 8 8 8  8 10 10 10
                1    9 9 9 9 9 9 10 10 10 10
"""
# 요건에 따라 우선순위별 제외 사유코드 생성
def treat_conditions(row):
    """ 우선순위별 제외사유코드 및 제외사유 생성 함수 """
    if row['B12000200'] >= 1:
        return 1
    elif row['B22000200'] >= 1:
        return 2
    elif row['BE0000801'] == 1:
        return 3
    elif row['DQ0052001'] >= 1:
        return 4
    elif row['P27000100'] >= 1:
        return 5
    else:
        return 0
pop1['제외사유코드'] = pop1.apply(treat_conditions, axis=1)
pop1['제외사유코드'].value_counts()

# 정상건만 추출하여 개발모집단 데이터셋(POP_DEV) 생성
pop_dev = pop1[pop1['제외사유코드'] == 0].copy()    # 문제 방지용

# TIP스코어등급, 중금리ML스코어등급 결합등급 생성
def treat_matrix_cond(row):
    """ matrix 형태의 조건 적용 함수 """
    condition_matrix = np.array(
        [[1,1,2,3,4,5, 6, 7, 8, 9],
         [1,2,2,3,4,5, 6, 7, 8, 9],
         [2,2,3,4,4,5, 6, 7, 8, 9],
         [3,3,4,4,5,5, 6, 7, 8, 9],
         [4,4,4,5,5,6, 6, 7, 8, 9],
         [5,5,5,5,6,6, 7, 7, 8, 9],
         [6,6,6,6,6,7, 7, 8, 8,10],
         [7,7,7,7,7,7, 8, 8,10,10],
         [8,8,8,8,8,8, 8,10,10,10],
         [9,9,9,9,9,9,10,10,10,10]])
    return condition_matrix[row[0]-1, row[1]-1]
score['결합등급'] = score[['ML0100_001_GRD','ML0100_002_GRD']].apply(treat_matrix_cond, axis=1)
score['결합등급'].value_counts().sort_index()
score[['ML0100_001_GRD','ML0100_002_GRD', '결합등급']]

""" 실습 5.2

    1) 실습 데이터: pop_dev
    2) 실습 과제:
        - pop_dev에 cps, score 데이터셋을 join하여 pop_dev1 데이터셋을 생성
        - 미리 생성된 항목을 활용하여 아래 항목들을 생성
            > 통합연체건수     = SUM( CB연체총건수,  대부업연체총건수,   기업여신연체총건수)
            > 통합최장연체일수 = MAX( CB최장연체일수, 대부업최장연체일수, 기업여신최장연체일수)
        - pop_dev1을 복사한 pop_dev2 데이터셋 생성
    3) 참고
	    - DQ0352001 : 대부업계 30일이상 연체 3개월내 경험한 총 건수
	    - DQ0352601 : 대부업계 30일이상 연체 3개월내 경험한 최장 연체경험기간
	    - DQ0652001 : 대부업계 30일이상 연체 6개월내 경험한 총 건수
	    - DQ0652601 : 대부업계 30일이상 연체 6개월내 경험한 최장 연체경험기간
	    - DQ1252001 : 대부업계 30일이상 연체 12개월내 경험한 총 연체건수
	    - DQ1252601 : 대부업계 30일이상 연체 12개월내 경험한 최장 연체 일수
	    - KC5000019 : 최근 3개월이내 연체 경험 과목수
	    - KC5000020 : 최근 6개월이내 연체 경험 과목수
	    - KC5000021 : 최근 12개월이내 연체 경험 과목수
	    - KC5000016 : 최근 3개월내 경험 최장 연체일수
	    - KC5000017 : 최근 6개월내 경험 최장 연체일수
	    - KC5000018 : 최근 12개월내 경험 최장 연체일수
	    - PHC000001 : 최근3개월내 경험한 연체 총 건수 (관리기간외 포함)
	    - PHC000007 : 최근6개월내 경험한 연체 총 건수 (관리기간외 포함)
	    - PHC000014 : 최근1년내 경험한 연체 총 건수 (관리기간외 포함)
	    - PE0300602 : 최근 3개월내 경험한 최장 연체경험기간(한편넣기)
	    - PE0600602 : 최근 6개월내 경험한 최장 연체경험기간(한편넣기)
	    - PE1200601 : 최근1년내 경험한 최장 연체경험기간(해제후1년경과건제외)(한편넣기)
"""

# pop_dev에 cps, score 데이터셋을 join하여 pop_dev1 데이터셋을 생성
# Key 중복 체크
pop_dev[['NO','stday']].duplicated().sum()  # T=1, F=0을 이용해 다 더했을 때 0이 나오면 중복 없음을 확인할 수 있음
cps[['NO','stday']].duplicated().sum()
score[['NO','stday']].duplicated().sum()
# join
pop_dev1 = pd.concat([pop_dev.set_index(['NO','stday']),
                      cps.set_index(['NO','stday']),
                      score.set_index(['NO','stday'])], axis=1, join='inner')
pop_dev1.isna().any().sum() # 검증하기
# 지금 상태에서는 중복도 없고 null값도 없는 상태임.

# 미리 생성된 항목을 활용하여 아래 항목들을 생성
# sum
pop_dev1['통합연체건수_3개월'] = pop_dev1['DQ0352001'] + pop_dev1['KC5000019'] + pop_dev1['PHC000001']
pop_dev1['통합연체건수_6개월'] = pop_dev1['DQ0652001'] + pop_dev1['KC5000020'] + pop_dev1['PHC000007']
pop_dev1['통합연체건수_12개월'] = pop_dev1['DQ1252001'] + pop_dev1['KC5000021'] + pop_dev1['PHC000014']
# max
pop_dev1['통합연체일수_3개월'] = pop_dev1[['DQ0352601','KC5000016','PE0300602']].max(axis=1)
pop_dev1['통합연체일수_6개월'] = pop_dev1[['DQ0652601','KC5000017','PE0600602']].max(axis=1)
pop_dev1['통합연체일수_12개월'] = pop_dev1[['DQ1252601','KC5000018','PE1200601']].max(axis=1)

# pop_dev1을 복사한 pop_dev2 데이터셋 생성
pop_dev2 = pop_dev1.reset_index().copy()
 


###############################################################################
# 6. 금융 데이터 분석 실습3
###############################################################################
""" 실습 6.1

    1) 실습 데이터: pop_dev2
    2) 실습 과제:
        - ID순번, 기준년월 기준 최종 기준일자 신청건 정보 생성
        - CB스코어(RK0600_000_SCR) 전체 평점분포, 직업군별 평점분포를 산출
        - TIP스코어등급(ML0100_001_GRD)과 중금리스코어등급(ML0100_002_GRD) 전체 및 직업군별 결합분포 산출
        - pop_dev2과 cb_delq_raw를 inner join하여 pop_dev3 데이터셋 생성
"""
# ID순번, 기준년월 기준 최종 기준일자 신청건 정보 생성
pop_dev2['YYMM'] = pop_dev2['STDAY_DT'].dt.strftime("%Y%m") # 기준년월 만듦
pop_dev2['STDAY_max'] = pop_dev2.groupby(['NO','YYMM'])['STDAY_DT'].transform('max')    # 그냥 max를 쓴다면 원래 행 개수 유지 X -> transform을 사용하면 행 개수 유지
pop_dev2['STDAY_cnt'] = pop_dev2.groupby(['NO','YYMM'])['STDAY_DT'].transform('count')
pop_dev2.loc[pop_dev2['STDAY_cnt']>1, ['NO', 'YYMM', 'STDAY_DT', 'STDAY_max']]  
pop_dev2 = pop_dev2[pop_dev2['STDAY_max'] == pop_dev2['STDAY_DT']].copy()   # 원하는 기준별로 한 건씩만 존재하되 같은 월에 여러번 신청된 것 중에 마지막것만 살아남게


# CB스코어(RK0600_000_SCR) 전체 평점분포, 직업군별 평점분포 산출
pop_dev2['RK0600_000_SCR'].value_counts()
pop_dev2['job_cd'] = pop_dev2['EW0020901'].astype(str).str[0]   # 숫자->문자열로 바꾸고 첫번째값만 가져오기
pop_dev2.groupby('job_cd')['RK0600_000_SCR'].value_counts() 

# TIP스코어등급(ML0100_001_GRD)과 중금리스코어등급(ML0100_002_GRD) 전체 및 직업군별 결합분포 산출
pop_dev2.pivot_table(index='ML0100_001_GRD',    
                     columns='ML0100_002_GRD',  
                     values='NO',   # 기준
                     aggfunc='count',   
                     fill_value=0)  # null은 0으로 채우기
pop_dev2.pivot_table(index=['job_cd', 'ML0100_001_GRD'],    # 인덱스 기준 2개
                     columns='ML0100_002_GRD',
                     values='NO',
                     aggfunc='count',
                     fill_value=0)

# pop_dev2과 cb_delq_raw를 inner join하여 pop_dev3 데이터셋 생성
# 중복 체크
pop_dev2[['NO','stday']].duplicated().sum() # 중복 없음
# join
pop_dev3 = pd.merge(pop_dev2, cb_delq_raw, on='NO', how='inner')    # 중복이 없는 데이터에 중복 추가해서 행 많이 만듦.




""" 실습 6.2

    1) 실습 데이터: pop_dev3
    2) 실습 과제:
        - 기준일자 이전 연체발생건 색인
        - 기준일자 이후 연체해제건 색인
        - 기연체(기준일자 시점에 이미 연체 중인 상황) 건 색인
        - 최장연체일수 항목 생성: 연체 시작일자, 연체 해제일자, 관측기간 고려
        - 최장연체일수 및 TPERF0017, TPERF0018 항목을 활용, TARGET 항목 생성
            > 불량 : 기준일 이후 365일내 최장연체일수 90일 이상 
                    또는 신용정보원 채불경험 
                    또는 CB 채불경험
        - 개발/검증 구분 항목 생성
            > 개발 : 기준년월 <= 2020년 2월
              검증 : 기준년월 => 2020년 3월

        - 개발/검증 구분별, 직업군별 불량률 산출

"""

# 기준일자 이전 연체발생건 색인 : 기준일(STDAY_DT) 이전에 연체가 시작된 이력이 있는지 확인
pop_dev3['이전연체발생건'] = np.where(pop_dev3['FST_OCR_DATE'] <= pop_dev3['STDAY_DT'], 1, 0)   # (1, 0)으로 색인
# 기준일자 이후 연체해제건 색인 : 기준일 시점에 연체가 진행중이고, 기준일 이후에 해제되는 케이스인지 확인
pop_dev3['이후연체종료건'] = np.where(pop_dev3['STDAY_DT'] < pop_dev3['REL_DATE_N'], 1, 0)
# 기연체(기준일자 시점에 이미 연체 중인 상황) 건 색인 : 기준일 시점에 이미 연체 중인 상태
pop_dev3['기연체'] = np.where(pop_dev3['이전연체발생건'] & pop_dev3['이후연체종료건'], 1, 0)
pop_dev3['기연체'].sum()    # 기연체 건수 확인
pop_dev3.loc[pop_dev3['기연체'] == 1, ['FST_OCR_DATE', 'STDAY_DT', 'REL_DATE_N']]   # 기연체 케이스가 정말 기준일을 관통하는지 샘플 확인

# 기준일자 이후 연체해제건 색인 : 연체가 기준일 이전에 이미 해제된 이벤트
pop_dev3['이전연체종료건'] = np.where(pop_dev3['REL_DATE_N'] <= pop_dev3['STDAY_DT'], 1, 0)
# 관심 외 연체건 제외
# - (기연체=1): 기준일 시점 연체 진행 중 케이스
# - (이전연체종료건=1): 기준일 이전에 이미 끝난 케이스
# -> 둘 다 제외하여 "기준일 이후 관측기간 내 성과 산출"에 집중
pop_dev3 = pop_dev3.loc[(pop_dev3['기연체'] == 0) & (pop_dev3['이전연체종료건'] == 0)]

# 최장연체일수 항목 생성: 연체 시작일자, 연체 해제일자, 관측기간 고려
pop_dev3['연체일수1'] = (pop_dev3['REL_DATE_N'] - pop_dev3['FST_OCR_DATE']).dt.days + 1 # 실제 해제일까지의 연체일수(해제일 기준)   /   +1 하는 이유: 날짜 차이가 0이어도 1일 연체로 카운트
pop_dev3['연체일수2'] = (pop_dev3['STDAY_AF_365D'] - pop_dev3['FST_OCR_DATE']).dt.days + 1  # 관측 종료일(기준일+365일)까지 이어졌다고 가정했을 때의 연체일수   /   해제가 관측기간 이후라면, 성과는 관측 종료일까지로 잘라서 계산해야 함
pop_dev3['연체일수'] = pop_dev3[['연체일수1','연체일수2']].min(axis=1)  # 관측기간을 고려한 실제 성과 연체일수  /   실제 해제일 기준 연체일수(연체일수1) vs 관측종료일 기준 연체일수(연체일수2) 중 더 작은 값을 선택하여 관측기간 밖 구간은 잘라냄

pop_dev3[['NO', 'STDAY_DT', 'FST_OCR_DATE', 'REL_DATE_N', 'STDAY_AF_365D', '연체일수1','연체일수2', '연체일수']]
pop_dev3['최장연체일수'] = pop_dev3.groupby(['NO', 'STDAY_DT'])['연체일수'].transform('max')    # 동일 고객(NO) + 동일 기준일(STDAY_DT)에서 여러 연체 이벤트가 있을 수 있으므로 관측기간 내 연체일수 중 최댓값
pop_dev3[['NO', 'stday', '최장연체일수']].drop_duplicates() 

# TARGET 항목 생성
pop_dev2['TARGET'] = np.where((pop_dev2['TPERF0017'] >= 1) |        # 신용정보원 채불 경험 >= 1
                              (pop_dev2['TPERF0018'] >= 1) |        # CB 채불 경험 >= 1
                              (pop_dev2['TPERF0013'] >= 90), 1, 0)  # 최장연체일수 >= 90
# 불량 = 위 중 하나라도 만족하면 1
pop_dev2['TARGET'].value_counts()

# 개발/검증 구분 항목 생성
pop_dev2['VAL_GB'] = np.where(pop_dev2['STDAY_DT'] < '2020-03-01', 0, 1)    # 개발 0, 검증 1
pop_dev2['VAL_GB'].value_counts()
pop_dev2.YYMM.value_counts().sort_index()

# 개발/검증 구분별, 직업군별 불량률 산출        
pop_dev2.pivot_table(index=['VAL_GB'],  # TARGET별 건수 집계
                     columns=['TARGET'],
                     values='NO',
                     aggfunc='count',
                     fill_value=0)
pop_dev2.pivot_table(index=['job_cd'],  # # 직업군별 TARGET 분포 집계
                     columns=['TARGET'],
                     values='NO',
                     aggfunc='count',
                     fill_value=0)



###############################################################################
# 7. 금융 데이터 분석 실습4
###############################################################################
""" 실습 7

    1) 실습 데이터: pop_dev2
    2) 실습 과제:
        - NEW CB스코어(RK0600_000_SCR)를 50단위로 구분
        - 개발/검증 구분별 전체 점수 분포 및 불량률 산출
"""
# NEW CB스코어(RK0600_000_SCR)를 50단위로 구분
bins = [-np.inf] + list(range(300, 1000, 50)) + [np.inf]
labels = [
    '300미만',
    '350미만',
    '400미만',
    '450미만',
    '500미만',
    '550미만',
    '600미만',
    '650미만',
    '700미만',
    '750미만',
    '800미만',
    '850미만',
    '900미만',
    '950미만',
    '950이상',
]
pop_dev2['RK0600_000_bin'] = pd.cut(pop_dev2['RK0600_000_SCR'], bins, right=False, labels=labels)   # right=False : 구간의 왼쪽 포함, 오른쪽 미포함
# pop_dev2[['RK0600_000_SCR', 'RK0600_000_bin']]
# pop_dev2.loc[pop_dev2['RK0600_000_SCR'] == 950, ['RK0600_000_SCR', 'RK0600_000_bin']]

# 개발/검증 구분별 전체 점수 분포 및 불량률 산출
# - VAL_GB: 개발(0) / 검증(1)
# - TARGET: 정상(0) / 불량(1)
pop_dev2.pivot_table(index=['VAL_GB','RK0600_000_bin'], 
                     columns=['TARGET'],
                     values='NO',
                     aggfunc='count',
                     fill_value=0)


